# 5. EDA espacial-temporal, variables y partici?n cronol?gica

Ejecutar despu?s de `0_ingesta_limpieza.ipynb`. Analiza distribuci?n por provincia/cant?n y estacionalidad; crea una tabla mensual para pron?stico y conserva un periodo final de validaci?n de enero a agosto de 2026. La unidad de predicci?n aqu? es **provincia-cant?n-mes**.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

ROOT=Path.cwd()
if not (ROOT/"data/processed").exists(): ROOT=ROOT.parent
OUT=ROOT/"data/processed"
df=pd.read_parquet(OUT/"incidentes_integrados.parquet")
df=df.loc[df.relevante_turismo].copy()
df["mes_periodo"]=df.fecha.dt.to_period("M")


## Concentraci?n geogr?fica y estacionalidad

El conteo por cant?n permite comparar concentraci?n administrativa, no densidad por ?rea ni coordenadas exactas. Para mapas coropl?ticos se necesita un GeoJSON oficial compatible. No se construye un mapa de puntos GPS porque el origen no trae coordenadas.

In [ ]:
por_canton=(df.groupby(["provincia","Canton"],dropna=False).size().rename("incidentes").reset_index().sort_values("incidentes",ascending=False))
por_provincia=(df.groupby("provincia",dropna=False).size().rename("incidentes").sort_values(ascending=False))
por_canton.head(20)


In [ ]:
estacionalidad=df.groupby(df.fecha.dt.month).size().reindex(range(1,13),fill_value=0)
ax=estacionalidad.plot(kind="bar",figsize=(10,4),color="#2563eb",title="Incidentes relevantes por mes calendario (todos los a?os)")
ax.set_xlabel("Mes"); ax.set_ylabel("Registros"); plt.tight_layout()
plt.show()
por_anio_mes=df.groupby([df.fecha.dt.year,df.fecha.dt.month]).size().rename("incidentes").reset_index()
por_anio_mes.to_csv(OUT/"eda_anio_mes.csv",index=False,encoding="utf-8-sig")


## Ingenier?a de variables y calendario

Los feriados nacionales est?n declarados como fechas editables de ejemplo y conviene mantenerlos desde una fuente oficial. Temporada alta es una hip?tesis configurable (feriados y fines de semana), no una etiqueta oficial de turismo. El clima mensual requiere una fuente meteorol?gica enlazada por regi?n y fecha; se deja fuera del entrenamiento hasta incorporarlo correctamente.

In [ ]:
# Agregaci?n mensual con cobertura expl?cita. Completa meses solo dentro de cada cant?n y marca exposici?n observada.
mensual=(df.groupby(["provincia","Canton", "mes_periodo"]).size().rename("target_incidentes").reset_index())
# calendario de fechas y variables temporales c?clicas
mensual["fecha_mes"]=mensual.mes_periodo.dt.to_timestamp()
mensual["anio"]=mensual.fecha_mes.dt.year
mensual["mes"]=mensual.fecha_mes.dt.month
mensual["mes_sin"]=np.sin(2*np.pi*mensual.mes/12)
mensual["mes_cos"]=np.cos(2*np.pi*mensual.mes/12)
# Fuente no contiene d?a/hora, as? que turno del d?a no es calculable.
feriados_ec=[]  # Ingresar fechas oficiales ISO: YYYY-MM-DD
feriados=set(pd.to_datetime(feriados_ec).date) if feriados_ec else set()
mensual["es_feriado"] = mensual.fecha_mes.dt.date.isin(feriados)
mensual["temporada_alta_proxy"] = mensual.fecha_mes.dt.month.isin([7,8,12]) | mensual.es_feriado
mensual["clima_promedio"] = np.nan  # completar con fuente externa validada, nunca imputar un valor ficticio
mensual.head()


## Divisi?n temporal y transformaci?n

Entrenamiento: hasta diciembre de 2025. Validaci?n: enero a agosto de 2026. Se omiten meses sin registros del agregado y deben revisarse contra `cobertura_mensual.csv`; el cuaderno de ingesta distingue ausencia de archivo de un cero real. El preprocesador se ajusta solo con train para evitar fuga de informaci?n.

In [ ]:
corte=pd.Timestamp("2026-01-01")
train=mensual.loc[mensual.fecha_mes<corte].copy()
valid=mensual.loc[(mensual.fecha_mes>=corte)&(mensual.fecha_mes<=pd.Timestamp("2026-08-31"))].copy()
print("Train",train.fecha_mes.min(),train.fecha_mes.max(),len(train))
print("Valid",valid.fecha_mes.min(),valid.fecha_mes.max(),len(valid))
cat_cols=["provincia","Canton"]
num_cols=["anio","mes_sin","mes_cos","es_feriado","temporada_alta_proxy","clima_promedio"]
preprocesador=ColumnTransformer([
 ("categoricas",Pipeline([("imputar",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),cat_cols),
 ("numericas",Pipeline([("imputar",SimpleImputer(strategy="median",add_indicator=True)),("escalar",StandardScaler())]),num_cols)
])
X_train=preprocesador.fit_transform(train[cat_cols+num_cols])
X_valid=preprocesador.transform(valid[cat_cols+num_cols])
y_train=train.target_incidentes.to_numpy(); y_valid=valid.target_incidentes.to_numpy()
print("Matriz",X_train.shape,X_valid.shape,"target: conteo mensual")
# Guardar datos interpretables; serializar preprocesador/modelo cuando exista un estimador entrenado.
train.to_parquet(OUT/"train_mensual.parquet",index=False)
valid.to_parquet(OUT/"valid_mensual.parquet",index=False)


## Lectura de resultados

La predicci?n de conteos por cant?n-mes no equivale a pron?stico de cada incidente ni a estimaci?n causal del peligro. Con pocos a?os de historia, compara un baseline estacional (mismo mes del a?o previo) con modelos m?s complejos y reporta MAE/RMSE adem?s de curvas de aprendizaje. Para prevenir underfitting, valida que las variables sean informativas; para controlar overfitting, eval?a exclusivamente el periodo futuro y evita ajustar el preprocesador con validaci?n.